# Notebook 4: Model Training, Evaluation & Interpretation

**Capstone Project — IoT Network Intrusion Detection on Imbalanced Data**

This notebook covers:
- **Task 5**: Train at least two ML models; evaluate with imbalanced-data metrics
- **Task 6**: Identify best pipeline; discuss trade-offs, misclassification implications, and limitations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score, roc_auc_score
)
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 1. Load Data and Feature Sets

In [ ]:
# Load processed data
df = pd.read_csv('processed_iot_intrusion.csv')

# Load selected features from Notebook 3
with open('selected_features.json', 'r') as f:
    feature_sets = json.load(f)

rfe_features = feature_sets['rfe_features']
chi2_features = feature_sets['chi2_features']
pca_n_components = feature_sets['pca_n_components']

print(f"Dataset shape: {df.shape}")
print(f"RFE features: {len(rfe_features)}")
print(f"Chi2 features: {len(chi2_features)}")
print(f"PCA components: {pca_n_components}")

In [ ]:
# Prepare data
X = df.drop(['label', 'label_encoded'], axis=1)
y = df['label_encoded']
label_names = df['label'].unique()

# Train-test split (consistent with previous notebooks)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training: {X_train.shape}, Test: {X_test.shape}")

## 2. Prepare Feature Sets and Sampling Strategies

We will cross-evaluate models across:
- **Sampling**: Original (no resampling) vs. SMOTE vs. SMOTETomek
- **Features**: RFE-selected features (best from Notebook 3)

In [ ]:
# Use RFE features (best performing from Notebook 3)
X_train_fs = X_train[rfe_features]
X_test_fs = X_test[rfe_features]

print(f"Using RFE features: {X_train_fs.shape[1]} features")

# Prepare sampling variants
print("\nPreparing sampling variants...")

# Original
datasets = {
    'Original': (X_train_fs, y_train)
}

# SMOTE
print("  Applying SMOTE...")
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train_fs, y_train)
datasets['SMOTE'] = (X_smote, y_smote)

# SMOTETomek
print("  Applying SMOTETomek...")
smotetomek = SMOTETomek(random_state=42)
X_st, y_st = smotetomek.fit_resample(X_train_fs, y_train)
datasets['SMOTETomek'] = (X_st, y_st)

for name, (X_s, y_s) in datasets.items():
    print(f"  {name}: {X_s.shape[0]:,} samples")

## 3. Define Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, multi_class='multinomial', solver='lbfgs', random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=20, random_state=42, n_jobs=-1
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=20, random_state=42
    )
}

# Optional: XGBoost if installed
try:
    from xgboost import XGBClassifier
    models['XGBoost'] = XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        eval_metric='mlogloss', random_state=42, n_jobs=-1,
        use_label_encoder=False
    )
    print("XGBoost available — included in evaluation.")
except ImportError:
    print("XGBoost not installed — proceeding with 3 models.")
    print("Install with: pip install xgboost")

print(f"\nModels to evaluate: {list(models.keys())}")

## 4. Train and Evaluate All Combinations

In [ ]:
all_results = []

for sampling_name, (X_s, y_s) in datasets.items():
    for model_name, model in models.items():
        print(f"Training {model_name} on {sampling_name} data...", end=' ')
        
        # Clone model to avoid state leakage
        from sklearn.base import clone
        m = clone(model)
        m.fit(X_s, y_s)
        y_pred = m.predict(X_test_fs)
        
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
        rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
        
        all_results.append({
            'Sampling': sampling_name,
            'Model': model_name,
            'Accuracy': round(acc, 4),
            'Precision': round(prec, 4),
            'Recall': round(rec, 4),
            'F1-Score': round(f1, 4)
        })
        
        print(f"Acc={acc:.4f}, F1={f1:.4f}")

results_df = pd.DataFrame(all_results)
print("\n=== Full Results ===")
results_df

## 5. Results Visualization

In [ ]:
# Grouped bar chart — F1-Score by Model and Sampling Strategy
pivot_f1 = results_df.pivot(index='Model', columns='Sampling', values='F1-Score')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

pivot_f1.plot(kind='bar', ax=axes[0], rot=15)
axes[0].set_title('Macro F1-Score by Model & Sampling', fontweight='bold')
axes[0].set_ylabel('Macro F1-Score')
axes[0].legend(title='Sampling')
axes[0].set_ylim(0, 1)

pivot_acc = results_df.pivot(index='Model', columns='Sampling', values='Accuracy')
pivot_acc.plot(kind='bar', ax=axes[1], rot=15)
axes[1].set_title('Accuracy by Model & Sampling', fontweight='bold')
axes[1].set_ylabel('Accuracy')
axes[1].legend(title='Sampling')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Best Model — Detailed Evaluation

In [ ]:
# Identify best model
best_row = results_df.loc[results_df['F1-Score'].idxmax()]
print("=== Best Performing Pipeline ===")
print(f"Model:    {best_row['Model']}")
print(f"Sampling: {best_row['Sampling']}")
print(f"Accuracy: {best_row['Accuracy']}")
print(f"F1-Score: {best_row['F1-Score']}")

In [ ]:
# Re-train the best model for detailed analysis
best_sampling = best_row['Sampling']
best_model_name = best_row['Model']

X_best, y_best = datasets[best_sampling]
best_model = clone(models[best_model_name])
best_model.fit(X_best, y_best)
y_pred_best = best_model.predict(X_test_fs)

# Full classification report
print(f"\n--- Classification Report ({best_model_name} + {best_sampling}) ---")
print(classification_report(y_test, y_pred_best))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=sorted(y.unique()), 
            yticklabels=sorted(y.unique()))
plt.title(f'Confusion Matrix — {best_model_name} + {best_sampling}', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('best_model_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Feature Importance (Best Model)

In [ ]:
# Feature importance (works for tree-based models)
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=rfe_features)
    importances_sorted = importances.sort_values(ascending=True)
    
    plt.figure(figsize=(10, 8))
    importances_sorted.plot(kind='barh', color='teal')
    plt.title(f'Feature Importance — {best_model_name}', fontsize=14, fontweight='bold')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nTop 10 Most Important Features:")
    print(importances.sort_values(ascending=False).head(10))
else:
    print("Feature importance not available for this model type.")

## 8. Precision-Recall Trade-off Analysis

In [ ]:
# Per-class precision and recall
from sklearn.metrics import precision_recall_fscore_support

prec_per_class, rec_per_class, f1_per_class, support = precision_recall_fscore_support(
    y_test, y_pred_best
)

pr_df = pd.DataFrame({
    'Class': sorted(y.unique()),
    'Precision': np.round(prec_per_class, 4),
    'Recall': np.round(rec_per_class, 4),
    'F1-Score': np.round(f1_per_class, 4),
    'Support': support
})

print("Per-Class Metrics:")
print(pr_df.to_string(index=False))

In [ ]:
# Precision vs Recall scatter
plt.figure(figsize=(10, 6))
scatter = plt.scatter(pr_df['Recall'], pr_df['Precision'], 
                      s=pr_df['Support'] / pr_df['Support'].max() * 500,
                      alpha=0.7, c=range(len(pr_df)), cmap='viridis', edgecolors='black')
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision vs Recall by Class (size = support)', fontsize=14, fontweight='bold')
plt.axhline(y=0.9, color='r', linestyle='--', alpha=0.3, label='90% threshold')
plt.axvline(x=0.9, color='r', linestyle='--', alpha=0.3)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('precision_recall_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Task 6: Interpretation and Discussion

### Best-Performing Pipeline
Based on the evaluation above, the best combination is identified by the highest Macro F1-Score.

### Trade-offs Between Recall and Precision
- **High Recall** is critical in intrusion detection — missing an attack (false negative) can lead to data breaches, device compromise, or service disruption.
- **High Precision** is also desirable to avoid overwhelming security teams with false alarms.
- SMOTE/SMOTETomek sampling generally improves recall for minority classes at a slight cost to precision on majority classes.
- The Precision vs Recall scatter plot above shows which classes achieve a good balance.

### Practical Implications of Misclassification
| Error Type | Impact | Cost |
|---|---|---|
| **False Negative** (attack classified as benign) | Attack goes undetected → potential breach | **Very High** |
| **False Positive** (benign classified as attack) | Unnecessary investigation triggered | Moderate |
| **Misclassified attack type** | Wrong response protocol activated | Moderate |

In a production IoT IDS, the system should be tuned for **high recall** (minimize false negatives), even if it means accepting a slightly higher false positive rate.

### Limitations
1. **Dataset specificity**: Trained on one IoT traffic dataset — may not generalize to different IoT environments or attack vectors.
2. **Static features**: The model uses precomputed flow statistics; real-time IDS would need streaming feature extraction.
3. **No temporal modeling**: Sequential patterns in network traffic are not captured by tabular classifiers.
4. **Computational cost**: SMOTE on 1M+ records is memory-intensive; SMOTETomek even more so.
5. **Concept drift**: Attack patterns evolve over time — the model would need periodic retraining.

### How the Solution Could Be Improved or Deployed
- **Deep Learning**: LSTM or 1D-CNN could capture temporal dependencies in traffic flows.
- **Ensemble stacking**: Combine Random Forest, XGBoost, and a neural network into a meta-learner.
- **Edge deployment**: Export the trained model as ONNX for lightweight inference on IoT gateways.
- **SHAP explainability**: Use SHAP values for per-prediction explanations to aid SOC analysts.
- **Continuous learning**: Implement an online learning pipeline that retrains on new traffic data.

## 9. Summary Results Table

In [ ]:
# Final summary
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
print(f"\nDataset: IoT Intrusion Detection ({df.shape[0]:,} records)")
print(f"Features: {len(rfe_features)} (RFE-selected)")
print(f"Classes: {y.nunique()} attack types")
print(f"\nBest Pipeline:")
print(f"  Model:    {best_row['Model']}")
print(f"  Sampling: {best_row['Sampling']}")
print(f"  Features: RFE ({len(rfe_features)} features)")
print(f"  Accuracy: {best_row['Accuracy']}")
print(f"  F1-Score: {best_row['F1-Score']}")
print("\n" + "="*60)

# Full comparison
print("\nAll Results:")
print(results_df.sort_values('F1-Score', ascending=False).to_string(index=False))